# Nyaya: build the index on a Kaggle GPU

Runs the full ingestion pipeline (Stages 1–7) on Kaggle and packages the result for the laptop.

**Before you click Run All** (right-hand panel → *Session options*):
- **Accelerator:** GPU T4 x2 or GPU P100.
- **Internet:** On. This needs a phone-verified Kaggle account. It's used to clone the repo, `pip install`, and download BGE-M3 / MiniLM / the reranker from Hugging Face.
- **Private repo only:** *Add-ons → Secrets → Add secret* named `GITHUB_TOKEN`, containing a GitHub token with read access to the repo. A public repo needs nothing.
  - *No-token alternative:* upload the repo as a Kaggle Dataset and attach it (*Add Input*); cell 2 then uses it instead of cloning.

**Output:** `/kaggle/working/nyaya_index.zip`. Download it from the *Output* tab, then unzip it at the repo root on the laptop. It contains `ingestion_output/`, `db/chroma_db/` and `retrieval_traces/`.

In [ ]:
# 1. Settings
REPO = "AshvinAA/Nyaya"      # GitHub owner/name
BRANCH = "main"
WORK = "/tmp/Nyaya"          # code + docs live here; only the zip goes to /kaggle/working
EMBED_BATCH = 32             # chunks per GPU forward pass (BGE-M3, fp32: fits a T4/P100)
SMOKE_QUERIES = [            # retrieval smoke test after the build (Stage 1 LLM off)
    "How many days of maternity leave does a woman worker get?",
    "গার্মেন্টস শ্রমিকের গ্রেড-৪ এর নিম্নতম মজুরি কত?",
    "What is the penalty for an illegal strike?",
    # Overlay probe: the 2026 Act amends s.196 (unfair labour practices
    # by workers). Stage 7 should report dual_layer_pairs > 0 here.
    "What are the unfair labour practices by workers under section 196?",
]

In [ ]:
# 2. Get the code: attached Kaggle Dataset if present, else git clone (token from Secrets if set)
import glob, os, shutil, subprocess

shutil.rmtree(WORK, ignore_errors=True)
attached = [p for p in glob.glob("/kaggle/input/*/") if os.path.isdir(os.path.join(p, "ingestion"))]
if attached:
    shutil.copytree(attached[0], WORK)
    print("using attached dataset:", attached[0])
else:
    token = None
    try:
        from kaggle_secrets import UserSecretsClient
        token = UserSecretsClient().get_secret("GITHUB_TOKEN")
    except Exception:
        pass                                   # no secret -> public clone
    auth = f"x-access-token:{token}@" if token else ""
    url = f"https://{auth}github.com/{REPO}.git"
    r = subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, url, WORK],
                       capture_output=True, text=True)
    print((r.stdout + r.stderr).replace(token or "\0", "***"))   # never echo the token
    assert r.returncode == 0, "clone failed - private repo? add the GITHUB_TOKEN secret"
    print(subprocess.run(["git", "-C", WORK, "log", "--oneline", "-1"],
                         capture_output=True, text=True).stdout)

n_pdf = len(glob.glob(f"{WORK}/docs/*.pdf"))
n_tx = len(glob.glob(f"{WORK}/docs/transcriptions/*/p*.txt"))
print(f"docs: {n_pdf} PDFs, {n_tx} transcript pages")
assert n_pdf >= 10 and n_tx == 58, "docs/ or docs/transcriptions/ incomplete"

In [ ]:
# 3. Dependencies. Kaggle ships torch + sentence-transformers + sklearn; add the rest.
#    chromadb is pinned to the laptop's version so the laptop can open the index.
!pip install -q chromadb==1.5.9 pdfplumber==0.11.10 langchain-ollama==1.1.0
import importlib.metadata as md_, torch
for pkg in ("torch", "sentence-transformers", "chromadb", "pdfplumber", "scikit-learn"):
    print(f"{pkg:22s}", md_.version(pkg))
print("CUDA:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")
assert torch.cuda.is_available(), "no GPU - set Accelerator to GPU in Session options"

In [ ]:
# 4. Full ingestion run (Stages 1-7). --strict-embed: fail rather than build a MiniLM index.
import subprocess, sys
cmd = [sys.executable, "-u", "-m", "ingestion.run_pipeline", "--full",
       "--embed-batch", str(EMBED_BATCH), "--strict-embed"]
proc = subprocess.Popen(cmd, cwd=WORK, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in proc.stdout:
    print(line, end="")
assert proc.wait() == 0, "ingestion failed - read the log above"

In [ ]:
# 5. Verify the build before trusting it
import collections, json
out = f"{WORK}/ingestion_output"
man = json.load(open(f"{out}/index_manifest.json"))
print(json.dumps({k: man.get(k) for k in ("model", "device", "batch_size", "dim",
                                          "n_indexed", "collection_count")}, indent=1))
assert man["model"] == "BAAI/bge-m3", "index was NOT built with BGE-M3"
assert man["collection_count"] == man["n_indexed"] > 0

val = json.load(open(f"{out}/validation_report.json"))["documents"]
for d in val:
    print(f"{'BLOCKED' if d['blocked'] else 'ok':8s} {d['doc'][:55]:55s} review={len(d['review'])}")
assert not any(d["blocked"] for d in val)

chunks = json.load(open(f"{out}/prose_chunks.json"))
live = [c for c in chunks if c.get("retrievable", True)]
print("\nretrievable chunks per document:")
for doc, n in collections.Counter(c["doc"] for c in live).most_common():
    print(f"  {n:5d}  {doc}")
tx = [c for c in live if c.get("text_source") == "manual_transcription"]
ov = [c for c in live if c.get("amends_section")]
print(f"\ntranscribed chunks indexed: {len(tx)}   overlay chunks with amends_section: {len(ov)}")
assert len(tx) > 200 and len(ov) > 150, "Bangla transcripts missing from the index"

In [ ]:
# 6. Retrieval smoke test (Stage 1 LLM expansion off: no Ollama on Kaggle)
import subprocess, sys
for q in SMOKE_QUERIES:
    print("=" * 100, "\nQUERY:", q)
    r = subprocess.run([sys.executable, "-m", "retrieval.cli", q, "--no-llm"],
                       cwd=WORK, capture_output=True, text=True)
    print(r.stdout[-6000:])
    if r.returncode != 0:
        print("STDERR:", r.stderr[-3000:])

In [ ]:
# 7. Package for the laptop
import os, shutil, zipfile
zpath = "/kaggle/working/nyaya_index.zip"
with zipfile.ZipFile(zpath, "w", zipfile.ZIP_DEFLATED) as z:
    for top in ("ingestion_output", "db", "retrieval_traces"):
        for root, _, files in os.walk(os.path.join(WORK, top)):
            for f in files:
                full = os.path.join(root, f)
                z.write(full, os.path.relpath(full, WORK))
print(f"{zpath}: {os.path.getsize(zpath) / 1e6:.1f} MB")
print("Download from the Output tab, then unzip at the repo root on the laptop.")